# Notebook 05: Data Cleaning Pipeline

**Time:** 30 minutes  
**Prerequisites:** Notebook 04 complete  
**Goal:** Build a complete pretraining data cleaning pipeline with deduplication, PII removal, and quality filtering

This notebook will:
1. Implement MinHash deduplication (same technique used by LLaMA 4, FineWeb)
2. Filter by language and remove PII
3. Remove repetitive n-grams and HTML noise
4. Run the full pipeline end-to-end on sample data
5. Discuss HuggingFace's DataTrove/FineWeb architecture

> **Why this matters:** Meta's LLaMA 4 was trained on 22-40 trillion tokens. HuggingFace's FineWeb dataset has 15 trillion tokens. The difference between a good and bad model often comes down to data cleaning. This notebook teaches you the exact techniques used in production.

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

import src.data_pipeline
importlib.reload(src.data_pipeline)
from src.data_pipeline import (
    minhash_deduplication,
    filter_by_language,
    strip_pii,
    remove_html_noise,
    remove_repetitive_ngrams,
    quality_filter,
    run_cleaning_pipeline,
    show_pipeline_summary,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 05")

✓ Gemini client initialized (gemini-3-flash-preview)
Setup complete -- ready for Notebook 05


---

## Part 1: Sample Dataset

We'll use a sample dataset that demonstrates common data quality issues:
- PII (emails, phone numbers, credit cards)
- Non-English text
- HTML noise
- Duplicate/near-duplicate content
- Repetitive spam patterns
- Short/low-quality text

In [2]:
print("=" * 65)
print("Experiment 1: Load Sample Dataset")
print("=" * 65)
print()

# Try to load the Class 3 test data
import pandas as pd

csv_path = os.path.join(parent_dir, 'test_data', 'data', 'Fake_Pretraining_Texts.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join(parent_dir, '..', 'MLE_in_Gen_AI-Course', 'Class3', 'test_data', 'data', 'Fake_Pretraining_Texts.csv')

if os.path.exists(csv_path):
    fake_texts = pd.read_csv(csv_path)
    raw_dataset = list(fake_texts['Raw Text'])
    print(f"Loaded {len(raw_dataset)} documents from CSV")
else:
    # Create sample dataset
    raw_dataset = [
        "Hello! Contact us at support@data.org or call 123-456-7890. Your credit card 4111111111111111 was declined.",
        "Hola! Este articulo esta completamente en espanol y no deberia estar en un dataset en ingles.",
        "<html><body><div><h1>Breaking News</h1><p>This is a major event!</p><nav>Home About Contact</nav></div></body></html>",
        "Buy now! Best product ever. Best product ever. Best product ever. Best product ever.",
        "Python 3.14 introduces several improvements including better error messages. Learn more on the official site.",
        "Python 3.14 introduces several improvements including better error messages. Learn more on the official docs.",
        "<div>For inquiries, email jane_doe@example.com or visit our site. Card number: 378282246310005.</div>",
        "Large Language Models are transforming the AI landscape with few-shot capabilities and emergent reasoning.",
        "ok",
        "Buy now! Best product ever. Best product ever. Best product ever.",
    ]
    print(f"Created sample dataset with {len(raw_dataset)} documents")

print("\nRaw documents:")
for i, doc in enumerate(raw_dataset):
    print(f"  [{i}] {doc[:80]}{'...' if len(doc) > 80 else ''}")

Experiment 1: Load Sample Dataset

Loaded 10 documents from CSV

Raw documents:
  [0] Hello! Contact us at support@data.org or call 123-456-7890. Your credit card 411...
  [1] Hola! Este artículo está completamente en español. Teléfono: 11-2222-3333
  [2] <html><body><div><h1>Breaking News</h1><p>This is a major event!</p></div><foote...
  [3] Buy now! Best product ever. Best product ever. Best product ever.
  [4] Python 3.14 introduces several improvements including better error messages. Lea...
  [5] Python 3.14 introduces several improvements including better error messages. Lea...
  [6] <div>For inquiries, email jane_doe@example.com or visit our site. Card number: 3...
  [7] Large Language Models are transforming the AI landscape with few-shot capabiliti...
  [8] 这是一个包含有用技术信息的中文段落。电话号码：010-12345678
  [9] Buy now! Best product ever. Best product ever. Best product ever.


---

## Part 2: Step-by-Step Cleaning

Let's apply each cleaning stage individually to see what it does.

In [3]:
print("=" * 65)
print("Experiment 2a: HTML Noise Removal")
print("=" * 65)
print()

step1 = [remove_html_noise(t) for t in raw_dataset]

# Show changes
for i, (raw, clean) in enumerate(zip(raw_dataset, step1)):
    if raw != clean:
        print(f"  [{i}] BEFORE: {raw[:100]}...")
        print(f"       AFTER:  {clean[:100]}...")
        print()

Experiment 2a: HTML Noise Removal

  [0] BEFORE: Hello! Contact us at support@data.org or call 123-456-7890. Your credit card 4111111111111111 was de...
       AFTER:  Hello! Contact us at support@data.org or call 123-456-7890. Your credit card 4111111111111111 was de...

  [2] BEFORE: <html><body><div><h1>Breaking News</h1><p>This is a major event!</p></div><footer>Contact us</footer...
       AFTER:  Breaking News This is a major event! Contact us...

  [6] BEFORE: <div>For inquiries, email jane_doe@example.com or visit our site. Card number: 378282246310005.</div...
       AFTER:  For inquiries, email jane_doe@example.com or visit our site. Card number: 378282246310005....



In [4]:
print("=" * 65)
print("Experiment 2b: Language Filtering")
print("=" * 65)
print()

step2 = filter_by_language(step1, target_lang="en")

removed = set(step1) - set(step2)
if removed:
    print("\nRemoved (non-English):")
    for r in removed:
        print(f"  - {r[:60]}...")

Experiment 2b: Language Filtering

  [LangFilter] Input: 10 documents (target: en)
  [LangFilter] Output: 7 kept (3 rejected)

Removed (non-English):
  - For inquiries, email jane_doe@example.com or visit our site....
  - Hola! Este artículo está completamente en español. Teléfono:...
  - 这是一个包含有用技术信息的中文段落。电话号码：010-12345678...


In [5]:
print("=" * 65)
print("Experiment 2c: MinHash Deduplication")
print("=" * 65)
print()

step3 = minhash_deduplication(step2, threshold=0.7)

print(f"\nBefore dedup: {len(step2)} docs")
print(f"After dedup:  {len(step3)} docs")

Experiment 2c: MinHash Deduplication

  [Dedup] Input: 7 documents (threshold=0.7)
  [Dedup] Output: 5 unique (2 duplicates removed)

Before dedup: 7 docs
After dedup:  5 docs


In [7]:
print("=" * 65)
print("Experiment 2d: PII Removal")
print("=" * 65)
print()

step4 = [strip_pii(t) for t in step3]

# Show PII changes
for i, (before, after) in enumerate(zip(step3, step4)):
    if before != after:
        print(f"  [{i}] BEFORE: {before[:80]}")
        print(f"       AFTER:  {after[:80]}")
        print()

Experiment 2d: PII Removal

  [0] BEFORE: Hello! Contact us at support@data.org or call 123-456-7890. Your credit card 411
       AFTER:  Hello! Contact us at [EMAIL] or call [PHONE]. Your credit card [CREDIT_CARD] was



In [8]:
print("=" * 65)
print("Experiment 2e: Repetitive N-gram Removal")
print("=" * 65)
print()

step5 = [remove_repetitive_ngrams(t) for t in step4]

for i, (before, after) in enumerate(zip(step4, step5)):
    if before != after:
        print(f"  [{i}] BEFORE: {before[:80]}")
        print(f"       AFTER:  {after[:80]}")
        print()

Experiment 2e: Repetitive N-gram Removal

  [2] BEFORE: Buy now! Best product ever. Best product ever. Best product ever.
       AFTER:  Buy now! Best product ever.



In [9]:
# TODO 1: Analyze individual pipeline stages

print("=" * 65)
print("TODO 1: Pipeline Stage Analysis")
print("=" * 65)
print()

todo1_reflection = """

- Which cleaning stage removed the most documents? Why?
The removal of HTML noise (Experiment 2a) removed the most documents. This is because several of the raw documents contained HTML tags and structures that were stripped out, leaving behind empty or very short text. For example, the document with the news article had a lot of HTML elements that were removed, resulting in a much shorter cleaned version.

- Were there any false positives (good content incorrectly removed)?
Yes, there were some instances where legitimate content was removed during the HTML noise removal process. This happened when the HTML tags were embedded within the text or when the stripping process inadvertently removed important information.


- How would you adjust the deduplication threshold for different use cases?
The deduplication threshold can be adjusted based on the desired balance between retaining similar content and removing duplicates. For use cases where diversity is important (e.g., training a generative model), a lower threshold (e.g., 0.5) might be used to remove more similar documents. For use cases where precision is key (e.g., information retrieval), a higher threshold (e.g., 0.9) might be preferred to retain more documents that are only slightly different.

"""

print(todo1_reflection)

TODO 1: Pipeline Stage Analysis



- Which cleaning stage removed the most documents? Why?
The removal of HTML noise (Experiment 2a) removed the most documents. This is because several of the raw documents contained HTML tags and structures that were stripped out, leaving behind empty or very short text. For example, the document with the news article had a lot of HTML elements that were removed, resulting in a much shorter cleaned version.

- Were there any false positives (good content incorrectly removed)?
Yes, there were some instances where legitimate content was removed during the HTML noise removal process. This happened when the HTML tags were embedded within the text or when the stripping process inadvertently removed important information.


- How would you adjust the deduplication threshold for different use cases?
The deduplication threshold can be adjusted based on the desired balance between retaining similar content and removing duplicates. For use cases where diversity 

---

## Part 3: Full Pipeline End-to-End

Now let's run the complete pipeline as a single function call.

In [10]:
print("=" * 65)
print("Experiment 3: Full Cleaning Pipeline")
print("=" * 65)
print()

pipeline_result = run_cleaning_pipeline(
    texts=raw_dataset,
    target_lang="en",
    dedup_threshold=0.7,
    save_path=os.path.join(outputs_dir, 'cleaned_data.json'),
)

print("\n\nCleaned documents:")
for i, doc in enumerate(pipeline_result['cleaned_texts']):
    print(f"  [{i}] {doc[:80]}{'...' if len(doc) > 80 else ''}")

Experiment 3: Full Cleaning Pipeline

PRETRAINING DATA CLEANING PIPELINE
Input: 10 documents

Stage 1: HTML noise removal
  Done (10 documents)

Stage 2: Language filtering
  [LangFilter] Input: 10 documents (target: en)
  [LangFilter] Output: 7 kept (3 rejected)

Stage 3: MinHash deduplication
  [Dedup] Input: 7 documents (threshold=0.7)
  [Dedup] Output: 5 unique (2 duplicates removed)

Stage 4: PII removal
  [PII] Processed 5 documents

Stage 5: Repetitive n-gram removal
  [N-gram] Processed 5 documents

Stage 6: Quality filtering
  [Quality] Input: 5 documents
  [Quality] Output: 3 kept (2 rejected)

--- Pipeline Summary ---
  Input:  10 documents
  Output: 3 documents
  Removal rate: 70.0%
  Saved to: ../outputs/cleaned_data.json


Cleaned documents:
  [0] Hello! Contact us at [EMAIL] or call [PHONE]. Your credit card [CREDIT_CARD] was...
  [1] Python 3.14 introduces several improvements including better error messages. Lea...
  [2] Large Language Models are transforming the AI la

In [11]:
# Before/after comparison
show_pipeline_summary(raw_dataset, pipeline_result['cleaned_texts'])

PIPELINE BEFORE/AFTER
  Metric                        Before      After     Change
  -------------------------------------------------------
  Documents                         10          3         -7
  Total characters                 968        356       -612
  Total words                      119         53        -66
  Char reduction                                       63.2%


---

## Part 4: DataTrove & FineWeb Architecture (2025)

HuggingFace's **DataTrove** is the production standard for processing LLM pretraining data at scale. It powers **FineWeb** (15 trillion tokens) and **FineWeb-2** (1000+ languages).

DataTrove pipeline stages:
1. **Read** -- Load from Common Crawl WARC files
2. **Extract** -- trafilatura for main content extraction
3. **Filter** -- URL blocklist, language, quality heuristics
4. **Deduplicate** -- MinHash LSH (same technique we used above!)
5. **Write** -- Save to parquet/jsonl format

The key insight: **our manual pipeline above mirrors exactly what DataTrove does at scale.**

In [12]:
# TODO 2: Discuss scaling and DataTrove

print("=" * 65)
print("TODO 2: Scaling Data Cleaning")
print("=" * 65)
print()

start = time.time()
response = client.generate(
    prompt="""I just built a data cleaning pipeline with these stages:
1. HTML removal (BeautifulSoup)
2. Language filtering (langdetect)
3. MinHash deduplication (datasketch)
4. PII removal (regex)
5. Repetitive n-gram removal
6. Quality filtering (min length, word count)

Now I want to understand how to scale this to billions of documents.

1. How does HuggingFace's DataTrove handle this at scale?
2. What additional cleaning steps does FineWeb use that I'm missing?
3. What are the biggest bottlenecks when processing Common Crawl data?
4. How would you parallelize MinHash deduplication across 96 crawl snapshots?

Be specific about implementation details.""",
    system="You are an expert in large-scale data processing for LLM pretraining.",
    max_tokens=2600,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")


TODO 2: Scaling Data Cleaning

⚡ Generating with Gemini...
Response in 12.2s
Model: gemini-3-flash-preview
Tokens: 181 in, 1387 out
Stop reason: FinishReason.STOP
Scaling a data pipeline from thousands to billions of documents requires moving away from "script-based" processing toward a distributed, streaming architecture. Your current stack (BeautifulSoup, `langdetect`, `datasketch`) is great for prototyping but will hit severe bottlenecks at the "Common Crawl" scale.

Here is the breakdown of how to scale to the level of FineWeb (15T+ tokens).

---

### 1. How HuggingFace’s DataTrove handles this
[DataTrove](https://github.com/huggingface/datatrove) is designed to be "framework-agnostic" (not relying on Spark/Ray) to minimize overhead.

*   **Streaming-First:** It doesn't load datasets into memory. It uses `fsspec` to stream data directly from S3/GCS.
*   **Pipeline Abstraction:** Every step (Filter, Extractor, Deduplicator) is a `PipelineStep`. These are chained together.
*   **Faul

In [13]:

todo2_reflection = """


- What additional cleaning steps would you add to your pipeline?
 I would consider adding a step for removing boilerplate content, such as navigation menus, footers, and advertisements, which can be common in web-scraped data. This could be done using a library like Boilerpipe or by training a custom model to identify and remove such content.

- How would you handle the scale difference between 10 documents and 10 billion?
To handle the scale difference, I would leverage distributed computing frameworks like Apache Spark or Dask to parallelize the processing across a cluster of machines. For storage, I would use a distributed file system like HDFS or cloud storage solutions that can handle large volumes of data. Additionally, I would implement efficient batching and streaming techniques to process the data in chunks rather than loading everything into memory at once.

- What was the most surprising thing you learned about DataTrove/FineWeb?
One surprising aspect of DataTrove and FineWeb is their use of advanced deduplication techniques that go beyond simple hashing. They utilize a combination of MinHash and more sophisticated similarity measures to identify near-duplicate content, which allows them to retain more unique documents while still removing redundant ones. Additionally, their ability to process and clean data at such a massive scale while maintaining high quality is impressive, showcasing the importance of efficient algorithms and infrastructure in large-scale data processing.


"""

print()
print(todo2_reflection)





- What additional cleaning steps would you add to your pipeline?
 I would consider adding a step for removing boilerplate content, such as navigation menus, footers, and advertisements, which can be common in web-scraped data. This could be done using a library like Boilerpipe or by training a custom model to identify and remove such content.

- How would you handle the scale difference between 10 documents and 10 billion?
To handle the scale difference, I would leverage distributed computing frameworks like Apache Spark or Dask to parallelize the processing across a cluster of machines. For storage, I would use a distributed file system like HDFS or cloud storage solutions that can handle large volumes of data. Additionally, I would implement efficient batching and streaming techniques to process the data in chunks rather than loading everything into memory at once.

- What was the most surprising thing you learned about DataTrove/FineWeb?
One surprising aspect of DataTrove and Fi

---

## Summary & Reflection

In [14]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'

full_reflection = f"""
### Part 1 - Pipeline Stage Analysis

{_todo1}

---

### Part 2 - Scaling with DataTrove

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="05",
    section_title="Data Cleaning Pipeline",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     1
Total input tokens:  181
Total output tokens: 1,387
Total cost:          $0.0213

Last 1 calls:
  1. [23:46:29] 3 -- 181in/1387out -- $0.0213


## Notebook 05 Complete!

**What you accomplished:**
- Built a complete data cleaning pipeline (dedup, PII, language, quality)
- Ran the pipeline end-to-end on sample data
- Learned about DataTrove/FineWeb production pipelines

**Key concepts:**
- MinHash LSH enables efficient near-duplicate detection at scale
- PII removal is critical for legal compliance and safety
- DataTrove/FineWeb use the same techniques we implemented, just at massive scale

**Next:** Open **Notebook 06: Text-to-Speech & Voice Synthesis**